# 2. Environmental observation-level trajectory embedding

질문: latent model 없이, raw RGB observation descriptor만으로 같은 trajectories의 진행 방향이나 branch-like separation이 보이는가?

이 단계는 VLM이 무엇을 학습했는지가 아니라, environment observation 자체에 branch-relevant visual cue가 있는지 보는 가장 낮은 level의 diagnostic입니다.

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

from pathlib import Path
import sys

repo_root = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "analysis").is_dir():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Run this notebook inside the TopoVLM repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import importlib
from analysis.code import hm3d_trajectory_notebook
hm3d_trajectory_notebook = importlib.reload(hm3d_trajectory_notebook)

DEFAULT_DATA_ROOT = hm3d_trajectory_notebook.DEFAULT_DATA_ROOT
load_observation_feature_rows = hm3d_trajectory_notebook.load_observation_feature_rows
pca_2d = hm3d_trajectory_notebook.pca_2d
plot_embedding_trajectories = hm3d_trajectory_notebook.plot_embedding_trajectories
select_scene_trajectory_records = hm3d_trajectory_notebook.select_scene_trajectory_records
trajectory_selection_summary = hm3d_trajectory_notebook.trajectory_selection_summary

DATA_ROOT = repo_root / DEFAULT_DATA_ROOT
records = select_scene_trajectory_records(DATA_ROOT, max_trajectories=6, min_steps=80, min_turns=5)
trajectory_selection_summary(records)


In [ ]:
features, rows = load_observation_feature_rows(records, data_root=DATA_ROOT, max_frames_per_trajectory=64)
coords, explained = pca_2d(features)
print("observation_features", features.shape, "pca_explained", explained.round(3).tolist())
fig, ax = plot_embedding_trajectories(
    coords,
    rows,
    records,
    title="Observation-level RGB trajectory PCA",
)
plt.show()


읽는 방법: Notebook 1과 같은 `Txx` labels/colors를 사용합니다. 선이 trajectory order를 보존하면 observation descriptor가 path progression을 담고 있다는 뜻입니다. 서로 다른 trajectories가 명확히 갈라지면, top-down branch 후보가 visual observation level에서도 어느 정도 구분된다는 evidence가 됩니다.